In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar

N = 37_000_000
GAMMA = 1 / 10
D = 10
WINDOW_LEN = 7
BETA_MIN = 0.01
BETA_MAX = 1.5
DT = 1.0

confirmed = pd.read_csv("Canada_confirmed.csv")

if "Date" in confirmed.columns:
    confirmed = confirmed.rename(columns={"Date": "date"})
elif "Province/State" in confirmed.columns:
    confirmed = confirmed.rename(columns={"Province/State": "date"})

confirmed["date"] = pd.to_datetime(confirmed["date"], format="%m/%d/%Y")
confirmed = confirmed[["date", "Total_confirmed"]].copy()
confirmed = confirmed.sort_values("date").reset_index(drop=True)

confirmed["new_cases_raw"] = (
    confirmed["Total_confirmed"].diff().fillna(0)
)

confirmed["new_cases_smooth"] = (
    confirmed["new_cases_raw"]
    .rolling(7, center=True, min_periods=1)
    .mean()
)


def estimate_S_I_R(confirmed_data, on_date):
    lookback_start = on_date - pd.Timedelta(days=D + 1)

    recent = confirmed_data[
        (confirmed_data["date"] >= lookback_start)
        & (confirmed_data["date"] <= on_date)
    ]

    daily_new = recent["Total_confirmed"].diff().dropna()
    I = daily_new.sum()

    today_row = confirmed_data[confirmed_data["date"] == on_date]
    total_today = today_row["Total_confirmed"].values[0]

    R = total_today - I
    S = N - I - R

    return float(S), float(I), float(R)


def sir_derivatives(S, I, R, beta):
    dS = -beta * S * I / N
    dI = beta * S * I / N - GAMMA * I
    dR = GAMMA * I
    return dS, dI, dR


def run_sir_euler(beta, S0, I0, R0, days):
    S_current = S0
    I_current = I0
    R_current = R0

    S_values = []
    I_values = []
    R_values = []
    predicted_new_cases = []

    for day in range(days):
        new_infections = beta * S_current * I_current / N
        predicted_new_cases.append(new_infections)

        dS, dI, dR = sir_derivatives(
            S_current, I_current, R_current, beta
        )

        S_next = S_current + DT * dS
        I_next = I_current + DT * dI
        R_next = R_current + DT * dR

        S_values.append(S_next)
        I_values.append(I_next)
        R_values.append(R_next)

        S_current = S_next
        I_current = I_next
        R_current = R_next

    return (
        np.array(S_values),
        np.array(I_values),
        np.array(R_values),
        np.array(predicted_new_cases),
    )


def fit_beta_for_week(S0, I0, R0, actual_smooth_cases):
    def loss(beta):
        S, I, R, predicted_cases = run_sir_euler(
            beta, S0, I0, R0, WINDOW_LEN
        )
        log_predicted = np.log(predicted_cases + 1)
        log_actual = np.log(actual_smooth_cases + 1)
        return np.sum((log_predicted - log_actual) ** 2)

    result = minimize_scalar(
        loss, bounds=(BETA_MIN, BETA_MAX), method="bounded"
    )
    return result.x


simulation_start = pd.Timestamp("2020-03-01")
simulation_end = pd.Timestamp("2023-03-09")

current_week_start = simulation_start

all_forecast_dates = []
all_predicted_cases = []
all_actual_cases = []

training_week_ends = []
fitted_betas = []

while True:
    train_start = current_week_start
    train_end = train_start + pd.Timedelta(days=WINDOW_LEN - 1)

    forecast_start = train_start + pd.Timedelta(days=WINDOW_LEN)
    forecast_end = forecast_start + pd.Timedelta(days=WINDOW_LEN - 1)

    if forecast_end > simulation_end:
        break

    training_initial_date = train_start - pd.Timedelta(days=1)

    S_fit0, I_fit0, R_fit0 = estimate_S_I_R(
        confirmed, training_initial_date
    )

    training_window = confirmed[
        (confirmed["date"] >= train_start)
        & (confirmed["date"] <= train_end)
    ]

    if len(training_window) != WINDOW_LEN:
        current_week_start += pd.Timedelta(days=WINDOW_LEN)
        continue

    actual_smooth = training_window["new_cases_smooth"].values

    beta_n = fit_beta_for_week(S_fit0, I_fit0, R_fit0, actual_smooth)

    forecast_initial_date = train_end

    S_forecast0, I_forecast0, R_forecast0 = estimate_S_I_R(
        confirmed, forecast_initial_date
    )

    beta_used_for_forecast = beta_n

    S_pred, I_pred, R_pred, predicted_cases = run_sir_euler(
        beta_used_for_forecast,
        S_forecast0,
        I_forecast0,
        R_forecast0,
        WINDOW_LEN,
    )

    forecast_window = confirmed[
        (confirmed["date"] >= forecast_start)
        & (confirmed["date"] <= forecast_end)
    ]

    if len(forecast_window) != WINDOW_LEN:
        break

    actual_next_week = forecast_window["new_cases_raw"].values
    forecast_dates = pd.date_range(forecast_start, forecast_end)

    all_forecast_dates.extend(forecast_dates)
    all_predicted_cases.extend(predicted_cases)
    all_actual_cases.extend(actual_next_week)

    training_week_ends.append(train_end)
    fitted_betas.append(beta_n)

    current_week_start += pd.Timedelta(days=WINDOW_LEN)


all_forecast_dates = pd.to_datetime(all_forecast_dates)
all_predicted_cases = np.array(all_predicted_cases)
all_actual_cases = np.array(all_actual_cases)

MAE = np.mean(np.abs(all_predicted_cases - all_actual_cases))
MSE = np.mean((all_predicted_cases - all_actual_cases) ** 2)
RMSE = np.sqrt(MSE)

print("Overall MAE:", MAE)
print("Overall MSE:", MSE)
print("Overall RMSE:", RMSE)

actual_smoothed = (
    pd.Series(all_actual_cases)
    .rolling(7, center=True, min_periods=1)
    .mean()
)

plt.figure(figsize=(14, 5))
plt.plot(
    all_forecast_dates,
    actual_smoothed,
    label="Actual daily cases (7-day average)",
)
plt.plot(
    all_forecast_dates,
    all_predicted_cases,
    label="Euler forecast using previous week's beta",
)
plt.xlabel("Date")
plt.ylabel("Daily new cases")
plt.title("SIR Forward Euler: Strict One-Week-Ahead Forecast")
plt.legend()
plt.grid()
plt.savefig("forecast_plot.png", dpi=150, bbox_inches="tight")
plt.show()

plt.figure(figsize=(14, 4))
plt.plot(training_week_ends, fitted_betas)
plt.xlabel("Week end date")
plt.ylabel("Beta")
plt.title("Weekly Estimated Beta")
plt.grid()
plt.savefig("beta_plot.png", dpi=150, bbox_inches="tight")
plt.show()
